In [ ]:
# =============================================================
# marketing_campaign.csv 전처리 (Colab)
# 팀 결정사항(데이터_전처리.md) 반영
# =============================================================
# 실행 전: Colab 좌측 파일 아이콘 -> marketing_campaign.csv 업로드
# 또는 아래 파일 업로드 위젯 사용

from google.colab import files
uploaded = files.upload()  # marketing_campaign.csv 선택

# -------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

df = pd.read_csv("marketing_campaign.csv", sep=";", encoding="utf-8-sig")
print(f"[0] 원본 적재: {df.shape[0]}행 x {df.shape[1]}열")
n_before = len(df)

# 단위경제성 상수 (컬럼 삭제 전에 값 보관 — 손익분기 계산에 계속 필요)
COST_PER_CONTACT = int(df["Z_CostContact"].iloc[0])   # 3
REVENUE_PER_ACCEPT = int(df["Z_Revenue"].iloc[0])     # 11
BREAKEVEN_RATE = COST_PER_CONTACT / REVENUE_PER_ACCEPT
print(f"    손익분기 반응률 = {COST_PER_CONTACT}/{REVENUE_PER_ACCEPT} = {BREAKEVEN_RATE:.1%}")

# -------------------------------------------------------------
# 1. 완전중복 처리 (2단계)
#    ID를 제외한 28개 컬럼이 일치하는 레코드가 다수 존재.
#    Recency/Income/Dt_Customer가 동시에 일치할 확률은 사실상 0
#    -> 우연이 아니라 적재 과정의 중복 삽입.
# -------------------------------------------------------------

# 1a. Response까지 완전히 같은 행 -> 한 개만 남김 (관련 358행 / 제거 182건)
dup_key_all = [c for c in df.columns if c != "ID"]
n_full_dup_rows = int(df.duplicated(subset=dup_key_all, keep=False).sum())
df = df.drop_duplicates(subset=dup_key_all, keep="first").reset_index(drop=True)
print(f"[1a] 완전중복(Response 포함) 제거: 관련 {n_full_dup_rows}행 중 "
      f"{n_before - len(df)}건 삭제 -> {len(df)}행")

# 1b. Response만 다른 경우 -> Response=1 행을 대표값으로 남기고 나머지 삭제
#     (팀 결정: 삭제 대신 "캠페인에 반응한 적이 있다"는 신호를 보존)
n_before_1b = len(df)
conf_key = [c for c in df.columns if c not in ("ID", "Response")]
df["_conf_grp_size"] = df.groupby(conf_key, dropna=False)["Response"].transform("nunique")
conflict_mask = df["_conf_grp_size"] > 1

# 상충 그룹 내에서 Response=1 우선, 동일 그룹 내 첫 행만 유지
df["_sort_key"] = -df["Response"]  # Response=1이 위로 오도록 정렬
df = df.sort_values("_sort_key").drop_duplicates(subset=conf_key, keep="first")
df = df.drop(columns=["_conf_grp_size", "_sort_key"]).reset_index(drop=True)

print(f"[1b] Response만 상충하는 중복: Response=1 행을 대표값으로 채택 "
      f"-> {n_before_1b}행 -> {len(df)}행 (삭제 {n_before_1b - len(df)}건)")
print(f"[1]  중복 처리 합계: {n_before}행 -> {len(df)}행 "
      f"(삭제 {n_before - len(df)}건, {(n_before - len(df)) / n_before:.1%})")

# -------------------------------------------------------------
# 2. 불필요한 상수 컬럼 제거
# -------------------------------------------------------------
df = df.drop(columns=["Z_CostContact", "Z_Revenue"])
print("[2] Z_CostContact, Z_Revenue 컬럼 삭제 (값은 위 BREAKEVEN_RATE로 보존)")

# -------------------------------------------------------------
# 3. Income 결측/이상치 처리
#    - 극단값(666666) 포함 전체를 중앙값으로 대체 (팀 결정: 전체 기준 중앙값)
#    - Income=7500이 12건 반복되나, 생년/학력/혼인상태/가입일이 전부 달라
#      중복 삽입이 아닌 하한값(설문 기본값 등)으로 판단 -> 그대로 유지, 주석만 남김
# -------------------------------------------------------------
df["Income_missing"] = df["Income"].isna().astype(int)
n_income_missing = int(df["Income_missing"].sum())

income_median = df["Income"].median()
df["Income"] = df["Income"].fillna(income_median)

n_income_outlier = int((df["Income"] > 200000).sum())
df.loc[df["Income"] > 200000, "Income"] = income_median

print(f"[3] Income: 결측 {n_income_missing}건 + 이상치(>200000) {n_income_outlier}건 "
      f"-> 전체 중앙값({income_median:,.0f})으로 대체")
print("    * Income=7500이 12건 존재하나 다른 모든 컬럼값이 서로 다름 "
      "-> 중복 삽입이 아닌 소득 하한값(설문 기본응답 등)으로 추정, 그대로 유지")

# -------------------------------------------------------------
# 4. Year_Birth -> Age 파생 (2014년 기준), 극단적 이상치 필터링
#    1893/1899/1900년생 3건은 2014년 기준 114~121세로 비현실적 -> 행 제거
# -------------------------------------------------------------
n_before_age = len(df)
df = df[df["Year_Birth"] >= 1940].reset_index(drop=True)
print(f"[4] Year_Birth<1940 (3건, 114~121세) 제거: {n_before_age} -> {len(df)}행")

df["Age"] = 2014 - df["Year_Birth"]

# 나중에 범주형으로 묶기 위한 연령대 구간 (필요 시 EDA/모델링 단계에서 사용)
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 30, 40, 50, 60, 70, 200],
    labels=["20대이하", "30대", "40대", "50대", "60대", "70대이상"],
)
print("    Age 파생 완료(2014년 기준), AgeGroup 범주형 컬럼 생성")

# -------------------------------------------------------------
# 5. Education 4개 범주로 정리 (2n Cycle -> Master 편입)
# -------------------------------------------------------------
df["Education"] = df["Education"].replace({"2n Cycle": "Master"})
print("[5] Education: 2n Cycle -> Master 편입, 잔존 범주:",
      sorted(df["Education"].unique()))

# -------------------------------------------------------------
# 6. Marital_Status 정리
#    Alone, Absurd, YOLO -> Single 로 병합 (표본 3+2+2=7건, 정상 범주와 병합)
#    Married, Together, Single, Divorced, Widow 는 그대로 유지
# -------------------------------------------------------------
marital_map = {"Alone": "Single", "Absurd": "Single", "YOLO": "Single"}
n_recode = int(df["Marital_Status"].isin(marital_map).sum())
df["Marital_Status"] = df["Marital_Status"].replace(marital_map)
print(f"[6] Marital_Status 재코딩 {n_recode}건 -> Single 병합, "
      f"잔존 범주: {sorted(df['Marital_Status'].unique())}")

# -------------------------------------------------------------
# 7. Dt_Customer 날짜형 변환
# -------------------------------------------------------------
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="mixed")
print("[7] Dt_Customer datetime 변환 완료, 범위:",
      df["Dt_Customer"].min().date(), "~", df["Dt_Customer"].max().date())

# -------------------------------------------------------------
# 8. 파생변수
#    - Children = Kidhome + Teenhome
#    - AnyAccepted = 과거 캠페인(1~5) 중 하나라도 수락 여부
# -------------------------------------------------------------
df["Children"] = df["Kidhome"] + df["Teenhome"]

cmp_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
            "AcceptedCmp4", "AcceptedCmp5"]
df["AnyAccepted"] = (df[cmp_cols].sum(axis=1) > 0).astype(int)

print("[8] 파생변수 생성: Children(Kidhome+Teenhome), "
      "AnyAccepted(과거 캠페인 1개 이상 수락 여부)")

# -------------------------------------------------------------
# 9. 저장 및 요약
# -------------------------------------------------------------
df.to_csv("marketing_clean.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print(f"최종: {df.shape[0]}행 x {df.shape[1]}열 "
      f"(원본 {n_before}행 대비 {(n_before - df.shape[0]) / n_before:.1%} 손실)")
print(f"결측 잔존: {int(df.isna().sum().sum())}건")
print(f"전체 반응률: {df['Response'].mean():.2%}  |  손익분기: {BREAKEVEN_RATE:.1%}")
ct = pd.crosstab(df["AnyAccepted"], df["Response"], normalize="index")
print(f"AnyAccepted=1 반응률: {ct.loc[1,1]:.2%}  /  =0: {ct.loc[0,1]:.2%}  "
      f"(격차 {ct.loc[1,1]-ct.loc[0,1]:.2%}p)")
print("=" * 60)

df.head()

# -------------------------------------------------------------
# 10. Colab에서 결과 파일 다운로드
# -------------------------------------------------------------
files.download("marketing_clean.csv")

Saving marketing_campaign.csv to marketing_campaign.csv
[0] 원본 적재: 2240행 x 29열
    손익분기 반응률 = 3/11 = 27.3%
[1a] 완전중복(Response 포함) 제거: 관련 358행 중 182건 삭제 -> 2058행
[1b] Response만 상충하는 중복: Response=1 행을 대표값으로 채택 -> 2058행 -> 2039행 (삭제 19건)
[1]  중복 처리 합계: 2240행 -> 2039행 (삭제 201건, 9.0%)
[2] Z_CostContact, Z_Revenue 컬럼 삭제 (값은 위 BREAKEVEN_RATE로 보존)
[3] Income: 결측 24건 + 이상치(>200000) 1건 -> 전체 중앙값(51,537)으로 대체
    * Income=7500이 12건 존재하나 다른 모든 컬럼값이 서로 다름 -> 중복 삽입이 아닌 소득 하한값(설문 기본응답 등)으로 추정, 그대로 유지
[4] Year_Birth<1940 (3건, 114~121세) 제거: 2039 -> 2036행
    Age 파생 완료(2014년 기준), AgeGroup 범주형 컬럼 생성
[5] Education: 2n Cycle -> Master 편입, 잔존 범주: ['Basic', 'Graduation', 'Master', 'PhD']
[6] Marital_Status 재코딩 6건 -> Single 병합, 잔존 범주: ['Divorced', 'Married', 'Single', 'Together', 'Widow']
[7] Dt_Customer datetime 변환 완료, 범위: 2012-07-30 ~ 2014-06-29
[8] 파생변수 생성: Children(Kidhome+Teenhome), AnyAccepted(과거 캠페인 1개 이상 수락 여부)

최종: 2036행 x 32열 (원본 2240행 대비 9.1% 손실)
결측 잔존: 0건
전체 반응률: 15.37%  |  손익분기: 27.3%
AnyAccepted

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================================================
# 이전 캠페인 효과(연관성) 검증 + 라그 구조 + 세그먼트 클러스터링
# ※ 인과추론이 아닌 연관성/예측 관점의 분석입니다.
#    (처치 노출 정보 부재로 인과 식별 자체가 불가능 - 별도 문서 참고)
# =============================================================

# =============================================================
# 0. 파일 로드 (원본 marketing_campaign.csv / 전처리본 marketing_clean.csv
#    둘 중 어느 것을 업로드해도 자동으로 인식해서 처리)
# =============================================================
from google.colab import files
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 200)

uploaded = files.upload()  # marketing_campaign.csv 또는 marketing_clean.csv 선택

fname = list(uploaded.keys())[0]  # 실제 업로드한 파일명 사용
print(f"업로드된 파일: {fname}")

# 원본은 세미콜론 구분자, 전처리본(pandas to_csv 기본값)은 콤마 구분자
try:
    df = pd.read_csv(fname, sep=";", encoding="utf-8-sig")
    if df.shape[1] < 5:  # 세미콜론이 안 맞으면 컬럼이 1개로 뭉쳐짐 -> 콤마로 재시도
        raise ValueError("세미콜론 파싱 실패")
except Exception:
    df = pd.read_csv(fname, sep=",", encoding="utf-8-sig")

IS_PREPROCESSED = "AnyAccepted" in df.columns
print(f"전처리 완료 파일 여부: {IS_PREPROCESSED}  (shape={df.shape})")

if not IS_PREPROCESSED:
    print("원본 파일 감지 -> 전처리 실행 (팀 결정사항 반영)")

    dup_key_all = [c for c in df.columns if c != "ID"]
    df = df.drop_duplicates(subset=dup_key_all, keep="first").reset_index(drop=True)

    conf_key = [c for c in df.columns if c not in ("ID", "Response")]
    df["_g"] = df.groupby(conf_key, dropna=False)["Response"].transform("nunique")
    df = (df.sort_values("Response", ascending=False)
            .drop_duplicates(subset=conf_key, keep="first")
            .drop(columns="_g").reset_index(drop=True))

    df = df.drop(columns=["Z_CostContact", "Z_Revenue"])

    income_median = df["Income"].median()
    df["Income"] = df["Income"].fillna(income_median)
    df.loc[df["Income"] > 200000, "Income"] = income_median

    df = df[df["Year_Birth"] >= 1940].reset_index(drop=True)
    df["Age"] = 2014 - df["Year_Birth"]

    df["Education"] = df["Education"].replace({"2n Cycle": "Master"})
    df["Marital_Status"] = df["Marital_Status"].replace(
        {"Alone": "Single", "Absurd": "Single", "YOLO": "Single"})
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="mixed")
    df["Children"] = df["Kidhome"] + df["Teenhome"]

    CMP = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]
    df["TotalAcceptedCmp"] = df[CMP].sum(axis=1)
    df["AnyAccepted"] = (df["TotalAcceptedCmp"] > 0).astype(int)

    MNT = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts",
           "MntSweetProducts", "MntGoldProds"]
    df["TotalMnt"] = df[MNT].sum(axis=1)
    df["TotalNumPurchases"] = df[["NumWebPurchases", "NumCatalogPurchases",
                                   "NumStorePurchases"]].sum(axis=1)
else:
    print("전처리 완료 파일 감지 -> 전처리 단계 건너뛰고 바로 분석 진행")
    CMP = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]
    if "TotalAcceptedCmp" not in df.columns:
        df["TotalAcceptedCmp"] = df[CMP].sum(axis=1)
    if "TotalMnt" not in df.columns:
        MNT = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts",
               "MntSweetProducts", "MntGoldProds"]
        df["TotalMnt"] = df[MNT].sum(axis=1)
    if "TotalNumPurchases" not in df.columns:
        df["TotalNumPurchases"] = df[["NumWebPurchases", "NumCatalogPurchases",
                                       "NumStorePurchases"]].sum(axis=1)

print(f"분석셋: {df.shape[0]}행 x {df.shape[1]}열")

BREAKEVEN = 3 / 11  # Z_CostContact/Z_Revenue (원본값, 컬럼은 삭제됨)

# =============================================================
# 1. 이전 캠페인 효과 — 단순 로지스틱 (연관성 검증, 인과 아님)
# =============================================================
print("\n" + "=" * 60)
print("[1] Response ~ AnyAccepted (단순 연관성)")
print("=" * 60)

X1 = sm.add_constant(df[["AnyAccepted"]].astype(float))
m1 = sm.Logit(df["Response"], X1).fit(disp=0)
print(m1.summary2().tables[1])
or1 = np.exp(m1.params["AnyAccepted"])
ci1 = np.exp(m1.conf_int().loc["AnyAccepted"]).values
print(f"OR = {or1:.2f}  (95% CI [{ci1[0]:.2f}, {ci1[1]:.2f}])")
print("※ 처치(발송) 여부 정보가 없어 인과효과로 해석 불가. 연관성으로만 보고.")

# =============================================================
# 2. 참여도 프록시 통제 로지스틱
#    "원래 참여도가 높은 사람" 프록시로 웹방문/최근성/지출액 통제
#    -> 통제 후에도 남는 연관성 크기 확인 (완전한 교란 제거는 아님)
# =============================================================
print("\n" + "=" * 60)
print("[2] 참여도 프록시 통제 로지스틱")
print("=" * 60)

# TotalAcceptedCmp = AnyAccepted의 상위 지표라 함께 넣으면 다중공선성 유발
# -> TotalAcceptedCmp만 사용 (횟수 자체를 연속형 신호로 반영)
X2 = sm.add_constant(df[["TotalAcceptedCmp", "NumWebVisitsMonth",
                          "Recency", "MntWines", "MntMeatProducts"]].astype(float))
m2 = sm.Logit(df["Response"], X2).fit(disp=0)
print(m2.summary2().tables[1])
or2 = np.exp(m2.params["TotalAcceptedCmp"])
print(f"TotalAcceptedCmp 1회당 OR = {or2:.2f}")
print(f"참고: corr(TotalAcceptedCmp, MntWines)="
      f"{df['TotalAcceptedCmp'].corr(df['MntWines']):.2f} "
      f"(다중공선성 확인용, 0.7 이상이면 주의)")

# =============================================================
# 3. 라그(자기회귀) 구조 — 예측 피처 엔지니어링 관점
#    ⚠ 가정: 컬럼 순서(Cmp1→...→Cmp5→Response)가 실제 캠페인 집행 순서와 일치
#    이 가정은 데이터 내 검증 불가(캠페인별 날짜 없음).
#    아래 Cmp3→Cmp4 구간에서 방향이 뒤집히는 현상은 이 가정이 틀렸을 가능성을
#    시사하므로, 라그 피처를 실제 예측모델에 쓸 경우 반드시 민감도 분석 병행.
# =============================================================
print("\n" + "=" * 60)
print("[3] 라그 구조 (컬럼순서=시간순서 가정, 검증 불가)")
print("=" * 60)

seq = CMP + ["Response"]
for i in range(len(seq) - 1):
    prev, nxt = seq[i], seq[i + 1]
    ct = pd.crosstab(df[prev], df[nxt], normalize="index")
    r1 = ct.loc[1, 1] if 1 in ct.index else np.nan
    r0 = ct.loc[0, 1] if 0 in ct.index else np.nan
    rr = r1 / r0 if r0 > 0 else np.inf
    flag = "  <- 방향 반전, 순서 가정 재검토 필요" if rr < 1 else ""
    print(f"{prev}=1 -> {nxt} 수락률 {r1:.1%}  |  {prev}=0 -> {r0:.1%}  (RR={rr:.2f}){flag}")

df["lag1"] = df["AcceptedCmp5"]  # Response 직전 캠페인으로 가정
Xl = sm.add_constant(df[["lag1"]].astype(float))
ml = sm.Logit(df["Response"], Xl).fit(disp=0)
print(f"\nResponse ~ lag1: OR={np.exp(ml.params['lag1']):.2f} "
      f"vs Response ~ 누적(AnyAccepted): OR={or1:.2f}")
print("-> 둘의 차이가 작다면 '최근 1회'보다 '누적된 반응성향'이 더 강한 신호")

print("\n" + "=" * 60)
print("[3-1] H4: 5개 캠페인 동시투입 - 개별 계수 비교 (최근성 효과 검증)")
print("=" * 60)
print("판정기준: Cmp5(직전 캠페인으로 가정) 계수가 5개 중 최댓값이면 H4 성립")

X4 = sm.add_constant(df[CMP].astype(float))
m4 = sm.Logit(df["Response"], X4).fit(disp=0)
h4_res = pd.DataFrame({
    "coef": m4.params, "p": m4.pvalues, "OR": np.exp(m4.params)
}).round(3).drop("const")
h4_res = h4_res.sort_values("coef", ascending=False)
print(h4_res)

top_cmp = h4_res.index[0]
if top_cmp == "AcceptedCmp5":
    print(f"\n-> H4 성립: AcceptedCmp5가 계수 최댓값")
else:
    print(f"\n-> H4 기각: 계수 최댓값은 {top_cmp} (OR={h4_res.loc[top_cmp,'OR']:.2f}), "
          f"AcceptedCmp5는 {list(h4_res.index).index('AcceptedCmp5')+1}위 "
          f"(OR={h4_res.loc['AcceptedCmp5','OR']:.2f})")
    print("   ※ 1위 캠페인이 [3]의 라그 방향반전 구간과 겹치면 '컬럼순서=시간순서' "
          "가정 자체가 틀렸을 가능성 -> 최근성 효과로 해석 금지, 원인 불명으로 보고")



print("\n" + "=" * 60)
print("[4] K-means 클러스터링")
print("=" * 60)

# 피처 유형 분리: 인구통계 + 행동, 참여이력(AnyAccepted 등)은 결과 유출 방지를
# 위해 클러스터링에서 제외하고 사후 프로파일링에서만 사용
CLUSTER_FEATURES = ["Age", "Income", "Children", "Recency", "TotalMnt",
                     "TotalNumPurchases", "NumWebVisitsMonth", "NumDealsPurchases"]
print("클러스터링 투입 피처:", CLUSTER_FEATURES)

Xc = df[CLUSTER_FEATURES].copy()
Xc_scaled = StandardScaler().fit_transform(Xc)

print("\nk별 관성/실루엣 (k 결정용):")
for k in range(2, 7):
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xc_scaled)
    sil = silhouette_score(Xc_scaled, km_tmp.labels_)
    print(f"  k={k}  inertia={km_tmp.inertia_:8.0f}  silhouette={sil:.3f}")

pca = PCA().fit(Xc_scaled)
print("\nPCA 누적설명분산:", np.round(np.cumsum(pca.explained_variance_ratio_), 3).tolist())
print("(피처 8개라 축소 효과가 크지 않으면 PCA 생략 가능 - 해석성 우선)")

K = 4  # 실루엣/해석성 절충으로 선택. 위 표 보고 조정 가능
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xc_scaled)
df["Segment"] = km.labels_

print(f"\n=== K={K} 세그먼트 프로파일 ===")
profile = df.groupby("Segment").agg(
    n=("ID", "size"), Age=("Age", "mean"), Income=("Income", "mean"),
    Children=("Children", "mean"), Recency=("Recency", "mean"),
    TotalMnt=("TotalMnt", "mean"), TotalNumPurchases=("TotalNumPurchases", "mean"),
    NumWebVisitsMonth=("NumWebVisitsMonth", "mean"),
    AnyAccepted_rate=("AnyAccepted", "mean"), Response_rate=("Response", "mean"),
).round(2)
print(profile)

# =============================================================
# 5. 세그먼트별 "이전 캠페인 연관성" 이질성
# =============================================================
print("\n" + "=" * 60)
print("[5] 세그먼트별 AnyAccepted-Response 연관성 이질성")
print("=" * 60)

rows = []
for seg, sub in df.groupby("Segment"):
    ct = pd.crosstab(sub.AnyAccepted, sub.Response, normalize="index")
    r1 = ct.loc[1, 1] if 1 in ct.index else np.nan
    r0 = ct.loc[0, 1] if 0 in ct.index else np.nan
    rows.append({"Segment": seg, "n": len(sub), "n_AnyAcc1": (sub.AnyAccepted == 1).sum(),
                 "resp_AnyAcc1": r1, "resp_AnyAcc0": r0, "gap": r1 - r0,
                 "AnyAcc1_흑자": r1 > BREAKEVEN})
het = pd.DataFrame(rows).set_index("Segment")
print(het.round(3))
print(f"\n손익분기({BREAKEVEN:.1%}) 기준 AnyAccepted=1이 흑자인 세그먼트:",
      het[het["AnyAcc1_흑자"]].index.tolist())

# =============================================================
# 6. 투자 우선순위 — 세그먼트 단독이 아닌 세그먼트 x AnyAccepted 교차 기준
#    (세그먼트 전체 발송은 대부분 손익분기 미달 -> 교차 조건으로 좁혀야 함)
# =============================================================
print("\n" + "=" * 60)
print("[6] 예산 배분 우선순위: Segment x AnyAccepted 교차")
print("=" * 60)

cross = df.groupby(["Segment", "AnyAccepted"]).agg(
    n=("ID", "size"), resp=("Response", "mean")
)
cross["expected_profit_per_contact"] = cross["resp"] * 11 - 3
cross["total_expected_profit"] = cross["n"] * cross["expected_profit_per_contact"]
cross = cross.sort_values("expected_profit_per_contact", ascending=False)
print(cross.round(2))
print("\n-> expected_profit_per_contact > 0 인 (Segment, AnyAccepted) 조합이 "
      "우선 투자 대상. n이 너무 작은 조합(<30 등)은 신뢰구간 넓으니 별도 표시 권장.")

# =============================================================
# 7. 가설 H1~H4 검증 요약
# =============================================================
print("\n" + "=" * 60)
print("[7] 가설 검증 요약")
print("=" * 60)

h2_retain_rate = or2 / or1
h3_gap_ratio = het["gap"].max() / het["gap"].min()

summary = pd.DataFrame([
    {"가설": "H1", "내용": "참여이력 -> Response 오즈 상승",
     "판정기준": "OR>1, p<.05",
     "결과": f"OR={or1:.2f} (95%CI [{ci1[0]:.2f},{ci1[1]:.2f}])",
     "판정": "성립" if (or1 > 1 and m1.pvalues["AnyAccepted"] < 0.05) else "기각"},
    {"가설": "H2", "내용": "프록시 통제 후 효과 잔존",
     "판정기준": "유의 + 잔존율>=50%",
     "결과": f"무보정 OR={or1:.2f} -> 통제후 OR={or2:.2f} (잔존율 {h2_retain_rate:.1%})",
     "판정": "성립(경계)" if h2_retain_rate >= 0.5 else "기각"},
    {"가설": "H3", "내용": "세그먼트 이질성 존재",
     "판정기준": "세그먼트간 gap 최대/최소 비율>=1.5배",
     "결과": f"최대 gap={het['gap'].max():.1%}p / 최소 gap={het['gap'].min():.1%}p "
             f"(비율 {h3_gap_ratio:.2f}배)",
     "판정": "성립" if h3_gap_ratio >= 1.5 else "기각"},
    {"가설": "H4", "내용": "Cmp5(직전)가 최대 개별계수 (최근성 효과)",
     "판정기준": "Cmp5=계수 최댓값",
     "결과": f"1위={top_cmp}(OR={h4_res.loc[top_cmp,'OR']:.2f}), "
             f"Cmp5는 {list(h4_res.index).index('AcceptedCmp5')+1}위",
     "판정": "성립" if top_cmp == "AcceptedCmp5" else "기각"},
])
print(summary.to_string(index=False))
print("\n※ H4 기각 시: 컬럼순서=캠페인순서 가정이 검증 안 된 상태이므로 "
      "'최근성 효과'로 표현하지 말고 '일부 캠페인의 강한 개별 연관성'으로만 보고")

df.to_csv("marketing_analysis2.csv", index=False, encoding="utf-8-sig")
files.download("marketing_analysis2.csv")

Saving marketing_clean.csv to marketing_clean (1).csv
업로드된 파일: marketing_clean (1).csv
전처리 완료 파일 여부: True  (shape=(2036, 32))
전처리 완료 파일 감지 -> 전처리 단계 건너뛰고 바로 분석 진행
분석셋: 2036행 x 35열

[1] Response ~ AnyAccepted (단순 연관성)
                Coef.  Std.Err.          z          P>|z|    [0.025    0.975]
const       -2.369160  0.089019 -26.614155  4.655293e-156 -2.543633 -2.194686
AnyAccepted  2.020517  0.132932  15.199640   3.555706e-52  1.759975  2.281059
OR = 7.54  (95% CI [5.81, 9.79])
※ 처치(발송) 여부 정보가 없어 인과효과로 해석 불가. 연관성으로만 보고.

[2] 참여도 프록시 통제 로지스틱
                      Coef.  Std.Err.          z         P>|z|    [0.025    0.975]
const             -3.109448  0.273829 -11.355436  6.969129e-30 -3.646143 -2.572753
TotalAcceptedCmp   1.348881  0.112352  12.005803  3.312343e-33  1.128675  1.569088
NumWebVisitsMonth  0.247993  0.034494   7.189390  6.508119e-13  0.180385  0.315601
Recency           -0.026922  0.002756  -9.767495  1.552432e-22 -0.032324 -0.021520
MntWines          -0.000050  0.000255

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# =============================================================
# 실행용 예측모델 비교: 인구통계 x TotalMnt x AnyAccepted
#
# 인구통계는 항상 포함(기본 정보량), TotalMnt/AnyAccepted를 각각
# 넣고 뺀 2x2 완전조합으로 어느 변수가 실제 타겟팅 이익에 얼마나
# 기여하는지 정량 비교한다.
# =============================================================

from google.colab import files
uploaded = files.upload()  # marketing_clean.csv 선택

# -------------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 220)

fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)
print(f"업로드된 파일: {fname}  shape={df.shape}")

MNT = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts",
       "MntSweetProducts", "MntGoldProds"]
df["TotalMnt"] = df[MNT].sum(axis=1)

dummies = pd.get_dummies(df[["Education", "Marital_Status"]], drop_first=True).astype(int)
demo_feat = ["Income", "Age", "Children"]
y = df["Response"]

# =============================================================
# 2x2 완전조합
#   인구통계는 항상 포함 (Income, Age, Children, Education, Marital_Status)
#   TotalMnt / AnyAccepted를 있음(O)/없음(X)으로 조합
# =============================================================
combos = {
    "① 인구통계만": [],
    "② 인구통계+TotalMnt": ["TotalMnt"],
    "③ 인구통계+AnyAccepted": ["AnyAccepted"],
    "④ 인구통계+TotalMnt+AnyAccepted": ["TotalMnt", "AnyAccepted"],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(X, y):
    aucs = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=3000).fit(X.iloc[tr], y.iloc[tr])
        aucs.append(roc_auc_score(y.iloc[te], clf.predict_proba(X.iloc[te])[:, 1]))
    return np.mean(aucs), np.std(aucs)

def targeting_profit(X, y, top_pct):
    clf = LogisticRegression(max_iter=3000).fit(X, y)
    proba = clf.predict_proba(X)[:, 1]
    n_target = int(len(y) * top_pct)
    idx = np.argsort(-proba)[:n_target]
    resp_rate = y.iloc[idx].mean()
    return resp_rate, n_target * (resp_rate * 11 - 3)

print("\n" + "=" * 90)
print(f"{'모델':35s} {'AUC':>14s} {'상위20%반응률':>12s} {'상위20%이익(＄)':>13s} "
      f"{'상위10%반응률':>12s} {'상위10%이익(＄)':>13s}")
print("=" * 90)

rows = []
for name, extra in combos.items():
    X = pd.concat([df[demo_feat + extra], dummies], axis=1)
    auc, auc_std = cv_auc(X, y)
    r20, p20 = targeting_profit(X, y, 0.20)
    r10, p10 = targeting_profit(X, y, 0.10)
    rows.append({"모델": name, "AUC": round(auc, 3), "AUC_std": round(auc_std, 3),
                 "상위20%반응률": r20, "상위20%이익": round(p20),
                 "상위10%반응률": r10, "상위10%이익": round(p10)})
    print(f"{name:35s} {auc:.3f}±{auc_std:.3f}  {r20:11.1%}  ＄{p20:9,.0f}  "
          f"{r10:11.1%}  ＄{p10:9,.0f}")

res = pd.DataFrame(rows)

# =============================================================
# 증분 기여도 분석 - TotalMnt / AnyAccepted 각각의 순수 기여도
# =============================================================
print("\n" + "=" * 60)
print("증분 기여도 분석 (① 인구통계만 대비)")
print("=" * 60)

base_auc = res.loc[0, "AUC"]
base_profit = res.loc[0, "상위20%이익"]

print(f"TotalMnt 단독 추가    : AUC {res.loc[1,'AUC']-base_auc:+.3f}   "
      f"상위20%이익 {res.loc[1,'상위20%이익']-base_profit:+,.0f}＄")
print(f"AnyAccepted 단독 추가 : AUC {res.loc[2,'AUC']-base_auc:+.3f}   "
      f"상위20%이익 {res.loc[2,'상위20%이익']-base_profit:+,.0f}＄")
print(f"둘 다 추가            : AUC {res.loc[3,'AUC']-base_auc:+.3f}   "
      f"상위20%이익 {res.loc[3,'상위20%이익']-base_profit:+,.0f}＄")

base_rate = df["Response"].mean()
base_all_profit = len(y) * (base_rate * 11 - 3)
print(f"\n참고) 전체 무차별 발송 기준선: ＄{base_all_profit:,.0f}")

print("\n" + "=" * 60)
print("결론")
print("=" * 60)
print("AnyAccepted 단독 기여가 TotalMnt 단독 기여보다 훨씬 크다.")
print("=> 예측/실행 목적에서는 TotalMnt만 넣고 AnyAccepted를 빼는 것은")
print("   가장 강력한 예측신호를 스스로 포기하는 선택이라 비효율적이다.")
print("=> 반대로 세그먼트(H3 설명용, 별도 트랙)에서는 AnyAccepted 및")
print("   그와 상관 높은 TotalMnt 계열 변수를 넣으면 순환논증 위험이 있어")
print("   제외하는 것이 맞다. 두 트랙은 서로 다른 규칙을 따른다.")

df.to_csv("marketing_model_comparison.csv", index=False)
files.download("marketing_model_comparison.csv")

Saving marketing_clean.csv to marketing_clean.csv
업로드된 파일: marketing_clean.csv  shape=(2036, 32)

모델                                             AUC     상위20%반응률    상위20%이익(＄)     상위10%반응률    상위10%이익(＄)
① 인구통계만                             0.682±0.022        34.4%  ＄      319        42.9%  ＄      348
② 인구통계+TotalMnt                     0.718±0.018        34.9%  ＄      341        47.3%  ＄      447
③ 인구통계+AnyAccepted                  0.768±0.019        43.7%  ＄      737        57.1%  ＄      667
④ 인구통계+TotalMnt+AnyAccepted         0.783±0.015        45.2%  ＄      803        55.7%  ＄      634

증분 기여도 분석 (① 인구통계만 대비)
TotalMnt 단독 추가    : AUC +0.036   상위20%이익 +22＄
AnyAccepted 단독 추가 : AUC +0.086   상위20%이익 +418＄
둘 다 추가            : AUC +0.101   상위20%이익 +484＄

참고) 전체 무차별 발송 기준선: ＄-2,665

결론
AnyAccepted 단독 기여가 TotalMnt 단독 기여보다 훨씬 크다.
=> 예측/실행 목적에서는 TotalMnt만 넣고 AnyAccepted를 빼는 것은
   가장 강력한 예측신호를 스스로 포기하는 선택이라 비효율적이다.
=> 반대로 세그먼트(H3 설명용, 별도 트랙)에서는 AnyAccepted 및
   그와 상관 높은 TotalMnt 계열 변수를 넣으면 순환논증 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>